In [ ]:
# Main simulation loop - run this cell to start generating events
# (Interrupt with Ctrl+C to stop)

cycle_count = 0
try:
    while True:
        cycle_count += 1
        cycle_start = time.time()
        print(f"\n=== Cycle {cycle_count} ===")
        
        # Phase 1: Warning (25 seconds)
        phase_start = time.time()
        publish_trigger_event("low", "rain")
        while time.time() - phase_start < WARNING_DURATION:
            elapsed = time.time() - phase_start
            remaining = WARNING_DURATION - elapsed
            if int(remaining) % 5 == 0 and remaining < WARNING_DURATION - 0.1:  # Every 5 sec
                print(f"  Warning phase: {remaining:.1f}s remaining")
            time.sleep(0.5)
        
        # Phase 2: Flood (15 seconds)
        phase_start = time.time()
        publish_trigger_event("high", "rain")
        while time.time() - phase_start < FLOOD_DURATION:
            elapsed = time.time() - phase_start
            remaining = FLOOD_DURATION - elapsed
            if int(remaining) % 5 == 0 and remaining < FLOOD_DURATION - 0.1:  # Every 5 sec
                print(f"  Flood phase: {remaining:.1f}s remaining")
            time.sleep(0.5)
        
        # Phase 3: Recovery (5 seconds)
        phase_start = time.time()
        print(f"  Recovery phase started...")
        while time.time() - phase_start < RECOVERY_DURATION:
            time.sleep(0.5)
        print(f"✓ Cycle complete. Next cycle in {WARNING_DURATION}s...\n")
        
except KeyboardInterrupt:
    print("\n\nSimulation stopped by user.")
finally:
    connector.disconnect()
    print("Disconnected from MQTT broker.")

In [ ]:
# Configuration for trigger cycle
CYCLE_DURATION = 45  # Total cycle: 25 warning + 15 flood + 5 recovery
WARNING_DURATION = 25
FLOOD_DURATION = 15
RECOVERY_DURATION = 5

def get_iso_timestamp():
    """Return current time as ISO 8601 string."""
    return datetime.now(timezone.utc).isoformat()

def publish_trigger_event(severity, event_type="rain"):
    """Publish a trigger event to MQTT."""
    evt = TriggerEvent(
        event=event_type,
        severity=severity,
        timestamp=get_iso_timestamp()
    )
    topic = make_topic(mqtt_cfg, "trigger")
    payload = json.dumps(evt.to_json())
    publisher.publish_json(topic, payload, qos=1)
    print(f"[{evt.timestamp}] Published: {severity.upper()} {event_type} event")
    return evt

print("Starting trigger simulation...")
print(f"Cycle: {WARNING_DURATION}s warning → {FLOOD_DURATION}s flood → {RECOVERY_DURATION}s recovery")
print()

## Simulation Loop: Trigger Flood Events

Cycle:
1. **Warning Phase** (25 sec): Issue low-severity "rain" warning
2. **Flood Phase** (15 sec): Escalate to high-severity flood
3. **Recovery Phase** (5 sec): Water recedes, return to low (implicit via observer simulation)

In [ ]:
# Connect MQTT client
connector = MqttConnector(mqtt_cfg, client_id_suffix="trigger")
connector.connect()

# Wait for connection
if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
else:
    print("✗ Failed to connect to MQTT broker")

publisher = MqttPublisher(connector)

## Connect to MQTT Broker

In [ ]:
import time
from datetime import datetime, timezone
import json

from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher, make_topic
from simulated_city.flood import TriggerEvent

# Load configuration from config.yaml
cfg = load_config()
mqtt_cfg = cfg.mqtt
flood_cfg = cfg.flood

print(f"MQTT Broker: {mqtt_cfg.host}:{mqtt_cfg.port}")
print(f"Base Topic: {mqtt_cfg.base_topic}")
print(f"Flood Config: {flood_cfg}")

## Setup: Load Configuration and Connect to MQTT

# Agent: Trigger (Flood Simulator)

This notebook simulates a flood trigger event with a predictable cycle:
- Every 30 seconds, a flood event occurs
- Warning issued 25 seconds in advance
- Flood duration: 15 seconds
- Water level rises to ~6-7 meters during flood, then recovers

This mimics a coastal flood scenario where early warning can be provided to evacuate.

# Agent Trigger
This notebook simulates a flood trigger source. It will publish `TriggerEvent` messages to MQTT.
Imports and logic will use `simulated_city` helpers and configuration.